# 3주차 시연 — 합성 트랙 미니 E2E
**AI Application · 스마트모빌리티학과 · 2026-09-16**

시뮬레이터 없이, 합성으로 그린 트랙 이미지로 **E2E 파이프라인 한 바퀴**를 돕니다.

| 단계 | 이 노트북에서 |
|---|---|
| ① 수집 | 렌더러 + 전문가 규칙으로 (이미지, 조향) 5,000쌍 생성 |
| ② 정제 | 조향 히스토그램(직진 편향), 주행 단위 분할 |
| ③ 학습 | TinyPilotNet, 학습 루프 5단계, 베스트 체크포인트 |
| ④ 배포 | ONNX 내보내기, onnxruntime 추론 지연 측정 |
| ⑤ 평가 | 오프라인 MAE·산점도·최악 사례 + **미니 폐루프 주행** |
| ⑥ 반복 | 마지막 실험: 상태 퍼짐을 바꿔 다시 학습 |

**Colab 생존 규칙**
- 결과는 전부 Google Drive(`AIApp/week03`)에 저장됩니다. 세션이 끊겨도 파일은 남습니다.
- 끊기면 **0번 셀부터** 다시 실행하세요. 1분이면 복귀합니다.
- CPU 런타임으로 충분합니다 (학습 셀 2~4분). GPU가 잡히면 더 빠를 뿐입니다.

로컬에서 실행하려면: `pip install numpy matplotlib torch onnxruntime` 후 그대로 실행 (Drive 대신 `./outputs` 에 저장).

In [ ]:
# Colab에는 onnxruntime이 없을 수 있어 자동 설치 (로컬은 이미 있으면 건너뜀)
import importlib.util, subprocess, sys
for pkg in ["onnx", "onnxruntime"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("패키지 확인 완료")

In [ ]:
# ============ 0. 재접속 시 여기부터 ============
import os, time, math, json
import numpy as np
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

# Drive 마운트 (Colab이 아니면 자동으로 건너뜀)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_DIR = Path("/content/drive/MyDrive/AIApp/week03")
except Exception:
    SAVE_DIR = Path("./outputs/week03")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("저장 폴더:", SAVE_DIR)
print("장치:", device, "| torch", torch.__version__)

## 1. 합성 트랙 렌더러 — 오늘의 시뮬레이터

차량 상태 3개로 66×200 카메라 이미지를 그립니다.

| 기호 | 의미 | 단위 |
|---|---|---|
| `d` | 차선 중앙으로부터의 측면 오프셋 (+는 오른쪽) | m (도로 반폭 = 0.5) |
| `psi` | 도로 방향 대비 헤딩 오차 (+는 오른쪽을 봄) | rad |
| `kappa` | 도로 곡률 (+는 우회전 도로) | 1/m |

먼저 이 세 상태와 카메라가 무엇을 뜻하는지 그림으로 봅니다 (왼쪽: 위에서 본 평면도, 오른쪽: 옆에서 본 측면도 — 이미지의 행이 어떻게 거리 `z` 로 바뀌는지).

In [ ]:
# 기하 그림 — 상태 (d, psi, kappa) 와 카메라 투영. 아래 렌더러 식을 그대로 그린 것 (숫자를 바꿔 보세요). 그림 안 글자는 Colab 폰트 문제로 영문
d0, psi0, kappa0 = 0.25, 0.15, 0.20          # 예시 상태: 중앙에서 오른쪽 0.25 m, 오른쪽을 0.15 rad 봄, 우회전 도로
F0, H0, HOR0 = 160.0, 0.2, 10                 # 초점거리(px), 카메라 높이(m), 지평선 행 — 아래 렌더러와 같은 값
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#52514e"
fig, (ax, bx) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw=dict(width_ratios=[1.15, 1]))

# ---- (좌) 평면도: 차량 좌표계. x = 옆(오른쪽 +), z = 앞
z = np.linspace(0, 3.0, 200)
xc = -d0 - z * np.tan(psi0) + 0.5 * kappa0 * z**2               # 차선 중앙의 옆 위치 x_c(z)
ax.fill_betweenx(z, xc - 0.5, xc + 0.5, color="#d9d9dc", lw=0)     # 도로 (반폭 0.5 m)
ax.plot(xc, z, "--", color="#c9a227", lw=2, label="lane center x_c(z)")
ax.plot(xc - 0.5, z, color="#9a9a9e", lw=1.5); ax.plot(xc + 0.5, z, color="#9a9a9e", lw=1.5)
ax.add_patch(plt.Rectangle((-0.08, -0.16), 0.16, 0.32, color=BLUE))  # 차량 (원점, 앞 = +z)
ax.annotate("", xy=(0, 0.9), xytext=(0, 0.16), arrowprops=dict(arrowstyle="-|>", color=BLUE, lw=2)); ax.text(0.04, 0.85, "vehicle heading", color=BLUE)
t = 0.9; ax.plot([0, -t * np.tan(psi0)], [0, t], color=ORANGE, lw=1.5, ls=":")   # 차량 위치에서의 도로 방향
th = np.linspace(np.pi / 2, np.pi / 2 + psi0, 30); ax.plot(0.55 * np.cos(th), 0.55 * np.sin(th), color=ORANGE, lw=1.5)
ax.text(-0.36, 0.56, "ψ", color=ORANGE, fontsize=14)
ax.annotate("", xy=(-d0, -0.05), xytext=(0, -0.05), arrowprops=dict(arrowstyle="<->", color=ORANGE, lw=1.5)); ax.text(-d0 / 2 - 0.06, -0.30, "d", color=ORANGE, fontsize=14)
z1 = 1.5; x1 = -d0 - z1 * np.tan(psi0) + 0.5 * kappa0 * z1**2
ax.plot([-1.3, x1], [z1, z1], color=MUTED, lw=1, ls=":"); ax.plot(x1, z1, "o", color=INK, ms=6)
ax.text(-1.28, z1 + 0.06, f"z = {z1} m", color=MUTED); ax.text(x1 + 0.06, z1 - 0.13, "x_c(z)", color=INK)
ax.text(xc[-1] + 0.08, 2.75, "κ > 0: right turn", color=MUTED)
fov = np.arctan(100 / F0)                                            # 카메라 시야 반각 (이미지 반폭 100 px)
for sgn in (-1, 1): ax.plot([0, sgn * 3.2 * np.tan(fov)], [0, 3.2], color=BLUE, lw=0.8, ls="--", alpha=0.6)
ax.text(1.55, 2.95, "camera FOV", color=BLUE, alpha=0.8)
ax.set_xlim(-1.4, 2.2); ax.set_ylim(-0.4, 3.2); ax.set_aspect("equal"); ax.set_xlabel("lateral x (m, right +)"); ax.set_ylabel("forward z (m)")
ax.set_title("(a) top view: state d, ψ, κ and x_c(z)", loc="left"); ax.legend(loc="upper left", frameon=False)

# ---- (우) 측면도: 이미지 행 ↔ 지면 거리 z = F·h / (row − 지평선행)
bx.axhline(0, color="#9a9a9e", lw=1.5); bx.fill_between([0, 3.2], -0.06, 0, color="#d9d9dc", lw=0)
bx.plot(0, H0, "s", color=BLUE, ms=9); bx.text(-0.12, 0.42, f"camera (h = {H0} m)", color=BLUE)
bx.plot([0.3, 0.3], [H0 - 0.2, H0 + 0.12], color=INK, lw=2); bx.text(0.32, H0 + 0.11, "image plane", color=INK, fontsize=9)
bx.axhline(H0, color=MUTED, lw=0.8, ls="--"); bx.text(2.35, H0 + 0.02, "horizon row (10)", color=MUTED, fontsize=9)
for row, c in ((20, ORANGE), (40, "#1baf7a")):
    zr = F0 * H0 / (row - HOR0)                                       # 그 행이 보는 지면 거리
    bx.plot([0, zr], [H0, 0], color=c, lw=1.5); bx.plot(zr, 0, "o", color=c, ms=6)
    bx.text(zr, -0.11, f"row {row} -> z = {zr:.2f} m", color=c, ha="center", fontsize=9)
bx.set_xlim(-0.2, 3.4); bx.set_ylim(-0.2, 0.55); bx.set_xlabel("forward z (m)"); bx.set_ylabel("height (m)")
bx.set_title("(b) side view: image row -> distance z = F·h / (row − horizon)", loc="left")
for a in (ax, bx): a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

카메라 앞 거리 `z` 에서 차선 중앙의 옆 위치는 `x_c(z) = -d - z·tan(psi) + 0.5·kappa·z²` 이고,
이를 핀홀 원근으로 투영해 픽셀로 옮깁니다. **전문가 규칙**은 중앙 복귀 + 곡선 선회:

$$\text{steer} = \mathrm{clip}\big(-(2.0\,d + 1.5\,\psi) + v\,\kappa / \omega_{\max},\ -1,\ 1\big)$$

(조향 $s\in[-1,1]$ 은 최대 요레이트 $\omega_{\max}$ 에 대한 비율입니다. 소형 차량 스케일: 도로 반폭 0.5 m, 속도 0.5 m/s, $\omega_{\max}$ = 1 rad/s.)

In [ ]:
H, W = 66, 200
F_PX, CAM_H, Y_HORIZON = 160.0, 0.2, 10   # 초점거리(px), 카메라 높이(m), 지평선 행
ROAD_HALF = 0.5                            # 도로 반폭(m) — 소형 차량 스케일
V, OMEGA_MAX, KAPPA_MAX, DT = 0.5, 1.0, 0.25, 0.1   # 속도(m/s), 최대 요레이트(rad/s), 최대 곡률(1/m), 시간 간격(s)

def render(d, psi, kappa, noise=0.0, rng=np.random):
    # 배경: 하늘(위) + 풀(아래)
    img = np.zeros((H, W, 3), dtype=np.float32)
    img[:Y_HORIZON+1] = (0.55, 0.70, 0.90)
    img[Y_HORIZON+1:] = (0.35, 0.55, 0.30)
    rows = np.arange(Y_HORIZON + 4, H)           # 지평선 아래 행들
    z = F_PX * CAM_H / (rows - Y_HORIZON)        # 각 행이 보는 거리(m)
    xc = -d - z * np.tan(psi) + 0.5 * kappa * z**2
    def to_px(x):                                 # 옆 위치(m) → 픽셀 열
        return W / 2 + F_PX * x / z
    left, right, center = to_px(xc - ROAD_HALF), to_px(xc + ROAD_HALF), to_px(xc)
    cols = np.arange(W)[None, :]
    road = (cols >= left[:, None]) & (cols <= right[:, None])
    lane_w = np.clip(F_PX * 0.04 / z, 1.0, 12.0)[:, None]   # 폭 4cm 차선을 원근 투영
    lane = (np.abs(cols - left[:, None]) < lane_w) | (np.abs(cols - right[:, None]) < lane_w)
    dash = ((np.abs(cols - center[:, None]) < lane_w * 0.6) & ((z[:, None] // 0.5).astype(int) % 2 == 0))
    block = img[rows]
    block[road] = (0.42, 0.42, 0.45)
    block[lane] = (0.95, 0.95, 0.95)
    block[dash] = (0.90, 0.80, 0.20)
    img[rows] = block
    if noise > 0:
        img += rng.normal(0, noise, img.shape).astype(np.float32)
    return (np.clip(img, 0, 1) * 255).astype(np.uint8)

def expert_steer(d, psi, kappa):
    # 중앙 복귀(피드백) + 곡선 선회(피드포워드). 조향 s∈[-1,1] 은 요레이트 s·OMEGA_MAX 로 이어짐
    return float(np.clip(-(2.0 * d + 1.5 * psi) + V * kappa / OMEGA_MAX, -1, 1))

# 확인: 상태 3가지를 그려보기
fig, axes = plt.subplots(1, 3, figsize=(12, 2.6))
for ax, (dd, pp, kk) in zip(axes, [(0, 0, 0), (0.25, 0.1, 0), (0, 0, 0.2)]):
    ax.imshow(render(dd, pp, kk)); ax.set_title(f"d={dd}, psi={pp}, kappa={kk} → steer={expert_steer(dd,pp,kk):+.2f}")
    ax.axis("off")
plt.tight_layout(); plt.show()

## 2. 데이터 생성 — '전문가가 달린 상태'만 샘플링

전문가는 중앙 근처를 달리므로 오프셋·헤딩은 0 근처에 몰려 있습니다. 이것이 나중에 **공변량 이동**의 씨앗이 됩니다.
`RUNS`개의 주행(run)으로 나누어 생성하고, 검증은 **주행 단위**로 떼어 둡니다.

In [ ]:
N_PER_RUN, RUNS = 500, 10        # 총 5,000장
GEN_NOISE = 0.02                 # 이미지 노이즈 (학습 이미지 생성에 사용)
STATE_SPREAD = 0.15              # 전문가 상태의 퍼짐(m) (클수록 가장자리 데이터가 많아짐)

def generate(n_per_run, runs, spread, noise, seed=0):
    rng = np.random.RandomState(seed)
    X = np.zeros((runs * n_per_run, H, W, 3), dtype=np.uint8)
    y = np.zeros(runs * n_per_run, dtype=np.float32)
    run_id = np.zeros(runs * n_per_run, dtype=np.int64)
    i = 0
    for r in range(runs):
        kappa = 0.0
        for _ in range(n_per_run):
            kappa = np.clip(kappa + rng.normal(0, 0.03), -KAPPA_MAX, KAPPA_MAX)   # 곡률이 천천히 변함
            d = np.clip(rng.normal(0, spread), -0.45, 0.45)
            psi = np.clip(rng.normal(0, spread * 1.2), -0.5, 0.5)
            X[i] = render(d, psi, kappa, noise, rng); y[i] = expert_steer(d, psi, kappa); run_id[i] = r
            i += 1
    return X, y, run_id

t0 = time.time()
X, y, run_id = generate(N_PER_RUN, RUNS, STATE_SPREAD, GEN_NOISE)
print(f"생성 완료: {X.shape}, {time.time()-t0:.1f}초")

fig, axes = plt.subplots(2, 5, figsize=(14, 3.8))
for ax, i in zip(axes.ravel(), np.random.choice(len(X), 10, replace=False)):
    ax.imshow(X[i]); ax.set_title(f"steer={y[i]:+.2f}", fontsize=9); ax.axis("off")
plt.suptitle("학습 데이터 샘플"); plt.tight_layout(); plt.show()

### 직진 편향 확인 + 주행 단위 분할

In [ ]:
plt.figure(figsize=(6, 3))
plt.hist(y, bins=41, color="#2563A8"); plt.xlabel("expert steer"); plt.ylabel("count")
plt.title("조향 라벨 분포 — 0 근처에 몰려 있는가?"); plt.tight_layout(); plt.show()
print(f"|steer| < 0.1 인 비율: {(np.abs(y) < 0.1).mean():.1%}")

# 주행(run) 단위 분할: run 0~7 학습, run 8~9 검증  (프레임을 섞어 나누면 데이터 누수!)
tr, va = run_id < 8, run_id >= 8
X_tr, y_tr, X_va, y_va = X[tr], y[tr], X[va], y[va]
print("학습:", X_tr.shape[0], "장 / 검증:", X_va.shape[0], "장")

## 3. 모델 — TinyPilotNet

NVIDIA PilotNet(2016)과 같은 골격: conv 5개 → FC 4개, 입력 3×66×200, 출력 조향 1개.
파라미터 약 25만 개 — 이 숫자가 "CPU로 되는 이유"입니다.

In [ ]:
class DriveDataset(Dataset):
    def __init__(self, images, steers):
        self.images, self.steers = images, steers
    def __len__(self):
        return len(self.steers)
    def __getitem__(self, i):
        img = torch.from_numpy(self.images[i]).permute(2, 0, 1).float() / 255.0   # (C,H,W), 0~1
        return img, torch.tensor([self.steers[i]], dtype=torch.float32)

class TinyPilotNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 24, 5, stride=2), nn.ReLU(),
            nn.Conv2d(24, 36, 5, stride=2), nn.ReLU(),
            nn.Conv2d(36, 48, 5, stride=2), nn.ReLU(),
            nn.Conv2d(48, 64, 3), nn.ReLU(),
            nn.Conv2d(64, 64, 3), nn.ReLU())
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(64 * 1 * 18, 100), nn.ReLU(),
            nn.Linear(100, 50), nn.ReLU(), nn.Linear(50, 10), nn.ReLU(), nn.Linear(10, 1))
    def forward(self, x):
        return self.head(self.features(x))

model = TinyPilotNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {n_params:,}")
with torch.no_grad():
    print("출력 shape 확인:", model(torch.zeros(2, 3, H, W).to(device)).shape)

## 4. 학습 루프 5단계

`forward → loss → zero_grad → backward → step`. 검증은 매 에폭, 저장은 **검증 손실이 최저일 때만** Drive에.

In [ ]:
EPOCHS, BATCH, LR = 10, 64, 1e-3
train_loader = DataLoader(DriveDataset(X_tr, y_tr), batch_size=BATCH, shuffle=True)
val_loader   = DataLoader(DriveDataset(X_va, y_va), batch_size=256, shuffle=False)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

history, best_val = {"train": [], "val": []}, float("inf")
t0 = time.time()
for epoch in range(EPOCHS):
    model.train(); run_loss, n = 0.0, 0
    for imgs, targets in train_loader:
        imgs, targets = imgs.to(device), targets.to(device)
        pred = model(imgs)                   # ① forward
        loss = criterion(pred, targets)      # ② loss
        optimizer.zero_grad()                # ③ zero_grad
        loss.backward()                      # ④ backward
        optimizer.step()                     # ⑤ step
        run_loss += loss.item() * len(imgs); n += len(imgs)
    model.eval(); val_loss, m = 0.0, 0
    with torch.no_grad():
        for imgs, targets in val_loader:
            val_loss += criterion(model(imgs.to(device)), targets.to(device)).item() * len(imgs); m += len(imgs)
    history["train"].append(run_loss / n); history["val"].append(val_loss / m)
    if val_loss / m < best_val:
        best_val = val_loss / m
        torch.save(model.state_dict(), SAVE_DIR / "best.pt")
    print(f"epoch {epoch+1}/{EPOCHS}  train={run_loss/n:.5f}  val={val_loss/m:.5f}  ({time.time()-t0:.0f}s)")
print("베스트 검증 MSE:", round(best_val, 5), "→", SAVE_DIR / "best.pt")

## 5. 학습 곡선 읽기

In [ ]:
plt.figure(figsize=(6, 3.2))
plt.plot(range(1, EPOCHS+1), history["train"], "o-", label="train", color="#2563A8")
plt.plot(range(1, EPOCHS+1), history["val"], "s-", label="val", color="#B05C10")
plt.xlabel("epoch"); plt.ylabel("MSE"); plt.legend(); plt.title("학습 곡선")
plt.tight_layout(); plt.savefig(SAVE_DIR / "learning_curve.png", dpi=120); plt.show()

## 6. 오프라인 평가 — MAE, 산점도, 최악 사례

'전문가가 갔던 상태'에서만 재는 지표입니다. 8번 셀의 폐루프 결과와 비교하세요.

In [ ]:
model.load_state_dict(torch.load(SAVE_DIR / "best.pt", map_location=device)); model.eval()
with torch.no_grad():
    preds = torch.cat([model(imgs.to(device)).cpu() for imgs, _ in val_loader]).numpy().ravel()
mae = np.abs(preds - y_va).mean()
print(f"검증 MAE = {mae:.4f}  (조향 -1~1 스케일)")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].scatter(y_va, preds, s=6, alpha=0.5, color="#2563A8"); axes[0].plot([-1, 1], [-1, 1], "k--", lw=1)
axes[0].set_xlabel("expert"); axes[0].set_ylabel("predicted"); axes[0].set_title("정답 vs 예측")
err = np.abs(preds - y_va)
axes[1].hist(err, bins=40, color="#B05C10"); axes[1].set_title("절대 오차 분포"); axes[1].set_xlabel("|error|")
plt.tight_layout(); plt.show()

worst = np.argsort(err)[-10:][::-1]
fig, axes = plt.subplots(2, 5, figsize=(14, 3.8))
for ax, i in zip(axes.ravel(), worst):
    ax.imshow(X_va[i]); ax.set_title(f"y={y_va[i]:+.2f} ŷ={preds[i]:+.2f}", fontsize=9); ax.axis("off")
plt.suptitle("최악 10장 — 어떤 상황에서 틀리는가?"); plt.tight_layout(); plt.show()

## 7. ONNX 내보내기 + 추론 지연

ONNX 파일 하나가 4주차 스타터 킷의 ROS2 추론 노드, 그리고 챌린지 제출물의 형식입니다.

In [ ]:
import onnxruntime as ort
onnx_path = SAVE_DIR / "pilot.onnx"
export_kw = dict(input_names=["image"], output_names=["steer"],
                 dynamic_axes={"image": {0: "batch"}}, opset_version=17)
try:   # torch 2.5+ : 새 exporter(onnxscript 필요) 대신 레거시 exporter 사용
    torch.onnx.export(model.cpu(), torch.zeros(1, 3, H, W), str(onnx_path), dynamo=False, **export_kw)
except TypeError:   # 구버전 torch 는 dynamo 인자가 없음
    torch.onnx.export(model.cpu(), torch.zeros(1, 3, H, W), str(onnx_path), **export_kw)
model.to(device)
sess = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])

def onnx_steer(img_uint8):
    x = (img_uint8.astype(np.float32) / 255.0).transpose(2, 0, 1)[None]   # 학습과 같은 전처리!
    return float(sess.run(None, {"image": x})[0][0, 0])

# 지연 측정
onnx_steer(X_va[0])
t0 = time.time(); [onnx_steer(X_va[i]) for i in range(200)]; dt = (time.time() - t0) / 200 * 1000
print(f"ONNX CPU 추론: {dt:.2f} ms/프레임  → 30fps 예산(33ms) 대비 {33/dt:.0f}배 여유")
print("파일 크기:", round(onnx_path.stat().st_size / 1024), "KB")

## 8. 미니 폐루프 — 모델이 직접 운전하면?

단순 운동학으로 차를 굴립니다: 조향이 헤딩을 바꾸고, 헤딩이 오프셋을 바꿉니다. 도로 곡률은 시간에 따라 변하고,
**50스텝마다 외란**(웅덩이·돌풍처럼 옆으로 0.35 m 밀리고 헤딩이 0.3 rad 튐)이 들어옵니다 — 전문가가 평소엔 가지 않는 상태입니다.
전문가 규칙과 우리 모델을 **같은 도로·같은 외란**에서 달리게 하고 오프셋 궤적을 비교합니다. `|d| > 0.5`(도로 반폭)이면 트랙 이탈.

In [ ]:
def rollout(policy, steps=400, seed=1, noise=GEN_NOISE, bump_every=50, bump_d=0.35, bump_psi=0.3):
    # bump_*: 주기적인 외란 — 웅덩이·돌풍처럼 차를 옆으로 밀고 헤딩을 튀게 함 (전문가가 가본 적 없는 상태 유발)
    rng = np.random.RandomState(seed)
    d, psi, kappa = 0.0, 0.0, 0.0
    traj, offtrack = [], 0
    for t in range(steps):
        kappa = np.clip(kappa + rng.normal(0, 0.03), -KAPPA_MAX, KAPPA_MAX)
        if bump_every and t > 0 and t % bump_every == 0:
            d = float(np.clip(d + rng.choice([-1, 1]) * bump_d, -ROAD_HALF, ROAD_HALF))
            psi = float(np.clip(psi + rng.choice([-1, 1]) * bump_psi, -1, 1))
        img = render(d, psi, kappa, noise, rng)
        s = float(np.clip(policy(img, d, psi, kappa), -1, 1))
        psi = psi + DT * (OMEGA_MAX * s - V * kappa)      # 조향은 헤딩을 돌리고, 도로 곡률은 상대 헤딩을 흘림
        psi = float(np.clip(psi, -1.0, 1.0))
        d = d + DT * V * math.sin(psi)
        traj.append(d)
        if abs(d) > ROAD_HALF:                            # 트랙 이탈 — 가장자리에 붙여 계속 진행
            offtrack += 1; d = float(np.clip(d, -ROAD_HALF, ROAD_HALF))
    return np.array(traj), offtrack

expert_policy = lambda img, d, psi, kappa: expert_steer(d, psi, kappa)   # 상태를 '아는' 전문가
model_policy  = lambda img, d, psi, kappa: onnx_steer(img)               # 이미지만 보는 모델

tr_e, off_e = rollout(expert_policy)
tr_m, off_m = rollout(model_policy)

plt.figure(figsize=(9, 3.2))
plt.plot(tr_e, label=f"expert (이탈 {off_e}회)", color="#2563A8")
plt.plot(tr_m, label=f"model (이탈 {off_m}회)", color="#B05C10")
plt.axhline(ROAD_HALF, color="k", ls="--", lw=0.8); plt.axhline(-ROAD_HALF, color="k", ls="--", lw=0.8)
plt.ylabel("lateral offset d (m)"); plt.xlabel("step"); plt.legend(); plt.title("폐루프 주행 — 오프셋 궤적")
plt.tight_layout(); plt.savefig(SAVE_DIR / "closed_loop.png", dpi=120); plt.show()
print(f"모델 평균 |d| = {np.abs(tr_m).mean():.3f}  (전문가 {np.abs(tr_e).mean():.3f})")

### 결과 해석
- 외란 직후 전문가는 몇 스텝 만에 중앙으로 돌아옵니다(상태를 정확히 알기 때문). 모델은 **이미지만 보고** 돌아와야 합니다.
- 모델이 외란 뒤 복귀하지 못하거나 이탈했다면 → **공변량 이동**의 예고편입니다. 밀려난 상태의 이미지는 학습 데이터에 (거의) 없었습니다.
- 잘 복귀했다면 → `STATE_SPREAD`가 충분히 커서 가장자리 상태를 학습 때 봤기 때문입니다. 아래 9번 실험에서 퍼짐을 줄이면 어떻게 되는지 확인하세요.

## 9. 마지막 실험 — 데이터를 바꾸면 폐루프가 바뀐다 (⑥ 반복)
아래 셀은 위 파이프라인을 함수로 묶은 것입니다. `spread` 를 바꿔 가며 폐루프 이탈 횟수를 비교하세요.

In [ ]:
def train_and_eval(spread, epochs=4, seed=0):
    Xs, ys, rid = generate(300, 8, spread, GEN_NOISE, seed)
    m = TinyPilotNet().to(device); opt = torch.optim.Adam(m.parameters(), lr=LR)
    loader = DataLoader(DriveDataset(Xs, ys), batch_size=BATCH, shuffle=True)
    for _ in range(epochs):
        m.train()
        for imgs, tg in loader:
            opt.zero_grad(); criterion(m(imgs.to(device)), tg.to(device)).backward(); opt.step()
    m.eval()
    def pol(img, d, psi, kappa):
        x = torch.from_numpy(img).permute(2, 0, 1).float().div(255).unsqueeze(0).to(device)
        with torch.no_grad():
            return float(m(x).cpu()[0, 0])
    _, off = rollout(pol)
    return off

for spread in [0.05, 0.15, 0.30]:
    print(f"STATE_SPREAD={spread:.2f} → 폐루프 이탈 {train_and_eval(spread)}회")

## 정리
- 파이프라인 6단계를 한 노트북에서 한 바퀴 돌았습니다: 생성 → 분할 → 학습 → ONNX → 오프라인 평가 → 폐루프 평가 → 데이터 바꿔 반복.
- **오프라인 지표와 폐루프 성능은 다르다.** 전문가가 간 상태만 배운 모델은 스스로 만든 상태에서 약합니다. 데이터의 '퍼짐'이 폐루프를 좌우했습니다.
- 다음 주: 스타터 킷의 `tinytrack` 이 이 렌더러를 패키지로 만든 것입니다. 진짜 데이터 수집과 Gazebo 시연으로 넘어갑니다.

**오늘 과제 없음.** 대신 준비 체크리스트(덱 마지막 장)를 다음 주까지 완료하세요.